# CLV Thesis — Fixed Validation Workflow (PRIMARY)

This notebook is now the primary thesis workflow after dropping HPO. It uses the pre-specified Valendin-style configuration set in `experiments/configs_final/`: fixed architecture defaults, 10% calibration-only validation for early stopping, 3 seeds, and 30 autoregressive scenarios.

Run the cells in order. The final holdout is used only for reporting, never for model selection. HPO cache/download cells have been removed from the primary path on purpose.


In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path

# ── 1. Install missing packages (skipped if already present) ─────────────────
# We do NOT touch numpy here. Kaggle's base image ships NumPy 2.x and ~15
# preinstalled packages (shap, jax, cupy, opencv, pytensor, ...) require >=2.0.
# The codebase was audited 2026-05-13 and is NumPy 2.x compatible.
def _need_install(pkg_name):
    try:
        __import__(pkg_name)
        return False
    except ImportError:
        return True

to_install = []
if _need_install("lifetimes"):  to_install.append("lifetimes>=0.11.3")
if _need_install("openpyxl"):   to_install.append("openpyxl>=3.1.0")

if to_install:
    print(f"Installing: {to_install}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + to_install, check=True)
    print("Done.\n")
else:
    print("All packages already installed.\n")

# ── 2. Set Kaggle environment flag ────────────────────────────────────────────
os.environ["KAGGLE_ENV"] = "1"
print("KAGGLE_ENV=1 set.\n")

# ── 3. Clone (or refresh) the repo from GitHub ────────────────────────────────
# Internet must be ON: Settings → Internet → On. Pin a specific commit by
# setting THESIS_REF before running this cell, e.g. os.environ["THESIS_REF"]="<sha>".
REPO_URL  = "https://github.com/OttoPrins/thesis-code-final.git"
REPO_PATH = Path("/kaggle/working/thesis-code")
REPO_REF  = os.environ.get("THESIS_REF", "main")

if REPO_PATH.exists():
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "--all", "--tags", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "checkout", REPO_REF, "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", f"origin/{REPO_REF}", "--quiet"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_PATH)], check=True)

os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))
sha = subprocess.check_output(
    ["git", "-C", str(REPO_PATH), "rev-parse", "--short", "HEAD"]
).decode().strip()
print(f"Repo : {REPO_PATH}  @ {sha}  (ref={REPO_REF})")


# ── 3.5. Ensure raw datasets are accessible ────────────────────────────────────
# CDNOW_sample.txt comes from git; CDNOW_master.txt is optional and fetched from Kaggle.
# UCI has a UCI ML Repository fallback URL if the Kaggle dataset upload was skipped.
# TaFeng and Dunnhumby come from Kaggle (pre-mounted symlink or kaggle CLI download).
DATA_ROOT = Path("/kaggle/working/input")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── CDNOW: sample from git, optional master from Kaggle ──────────────────
cdnow_dst    = DATA_ROOT / "cdnow-dataset"
cdnow_sample = cdnow_dst / "CDNOW_sample.txt"
cdnow_master = cdnow_dst / "CDNOW_master.txt"
cdnow_mount  = Path("/kaggle/input/cdnow-dataset")
cdnow_dst.mkdir(exist_ok=True)

# CDNOW_sample.txt (2,357-customer 10% sample) is tracked in git — copy from the
# repo. Fall back to the Kaggle dataset mount if the repo copy is missing.
if not cdnow_sample.exists():
    repo_sample = REPO_PATH / "data" / "raw" / "CDNOW_sample.txt"
    if repo_sample.exists():
        shutil.copy(repo_sample, cdnow_sample)
        print("  cdnow-dataset: CDNOW_sample.txt copied from git repo.")
    elif (cdnow_mount / "CDNOW_sample.txt").exists():
        shutil.copy(cdnow_mount / "CDNOW_sample.txt", cdnow_sample)
        print("  cdnow-dataset: CDNOW_sample.txt copied from /kaggle/input/.")
    else:
        print("  cdnow-dataset: WARN — CDNOW_sample.txt not found in repo or /kaggle/input/.")
else:
    print("  cdnow-dataset: CDNOW_sample.txt already present.")

# CDNOW_master.txt (23,570-customer cohort) is gitignored — comes from the Kaggle
# dataset only. Needed for the final CDNOW training configs.
if not cdnow_master.exists():
    if (cdnow_mount / "CDNOW_master.txt").exists():
        shutil.copy(cdnow_mount / "CDNOW_master.txt", cdnow_master)
        print("  cdnow-dataset: CDNOW_master.txt copied from /kaggle/input/.")
    else:
        print("  cdnow-dataset: CDNOW_master.txt not in /kaggle/input/ — trying kaggle download ...")
        try:
            subprocess.run(
                ["kaggle", "datasets", "download", "-d", "ottoprins/cdnow-dataset",
                 "-p", str(cdnow_dst), "--unzip", "--force"],
                check=True, capture_output=True, timeout=180,
            )
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
            print(f"  cdnow-dataset: kaggle download fallback failed ({type(e).__name__}).")
        if cdnow_master.exists():
            print("  cdnow-dataset: CDNOW_master.txt downloaded from ottoprins/cdnow-dataset.")
        else:
            print("  cdnow-dataset: WARN — CDNOW_master.txt still missing. "
                  "Final configs require CDNOW_master.txt; preflight will fail if it is missing.")
else:
    print("  cdnow-dataset: CDNOW_master.txt already present.")

# ── UCI: Kaggle download with UCI ML Repo fallback ────────────────────────────
uci_dst = DATA_ROOT / "uci-retail"
if uci_dst.exists():
    print("  uci-retail: already present.")
elif Path("/kaggle/input/uci-retail").exists():
    uci_dst.symlink_to(Path("/kaggle/input/uci-retail"))
    print("  uci-retail: symlinked from /kaggle/input/.")
else:
    print("  uci-retail: not mounted — trying kaggle download ...")
    try:
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", "ottoprins/uci-retail",
             "-p", str(uci_dst), "--unzip"],
            check=True, capture_output=True, timeout=300,
        )
        print("  uci-retail: kaggle download complete.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        print(f"  uci-retail: kaggle download failed ({type(e).__name__}) — fetching from UCI ML Repo ...")
        uci_dst.mkdir(exist_ok=True)
        subprocess.run(["wget", "-q", "-O", "/tmp/uci.zip",
            "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"],
            check=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/uci.zip", "-d", str(uci_dst)],
            check=True)
        # UCI zip may extract with a slightly different name — normalise it
        for f in sorted(uci_dst.iterdir()):
            if f.suffix in (".xlsx", ".csv") and "retail" in f.name.lower():
                target_name = uci_dst / "online_retail_II.xlsx"
                if not target_name.exists():
                    f.rename(target_name)
                break
        print("  uci-retail: UCI fallback download complete.")

# ── TaFeng and Dunnhumby: standard Kaggle download / symlink ─────────────────
for slug, api_ref in [("tafeng-dataset", "ottoprins/tafeng-dataset"),
                       ("dunnhumby",      "ottoprins/dunnhumby")]:
    target  = DATA_ROOT / slug
    mounted = Path(f"/kaggle/input/{slug}")
    if target.exists():
        print(f"  {slug}: already present.")
    elif mounted.exists():
        target.symlink_to(mounted)
        print(f"  {slug}: symlinked from /kaggle/input/.")
    else:
        print(f"  {slug}: not mounted — downloading ...")
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", api_ref,
             "-p", str(target), "--unzip"],
            check=True,
        )
        print(f"  {slug}: download complete.")

# ── Confirm what's on disk ────────────────────────────────────────────────────
print("\n── Data directory contents ──")
for slug in ["cdnow-dataset", "uci-retail", "tafeng-dataset", "dunnhumby"]:
    d = DATA_ROOT / slug
    if d.exists():
        files = sorted(f.name for f in d.iterdir() if not f.name.startswith("."))[:6]
        print(f"  {slug}: {files}")
    else:
        print(f"  {slug}: [MISSING]")

os.environ["KAGGLE_DATA_ROOT"] = str(DATA_ROOT)
print(f"\nKAGGLE_DATA_ROOT={DATA_ROOT}  — all datasets ready.\n")


# ── 3.7. Mirror input data into repo data/raw/ so pipeline configs resolve ──
# All final configs use raw_dir: data/raw (or data/raw/Dunnhumby datasets).
# Pipelines never read KAGGLE_DATA_ROOT directly — they always resolve raw_dir
# relative to CWD. Symlink/copy files from DATA_ROOT into the repo tree here.
repo_raw = REPO_PATH / "data" / "raw"
repo_raw.mkdir(parents=True, exist_ok=True)

# CDNOW_master.txt — small text file, copy is fine
_src = cdnow_master
_dst = repo_raw / "CDNOW_master.txt"
if not _dst.exists() and _src.exists():
    shutil.copy(_src, _dst)
    print("  data/raw/: CDNOW_master.txt synced from input.")

# UCI Online Retail II — symlink to avoid duplicating a large xlsx
_src = DATA_ROOT / "uci-retail" / "online_retail_II.xlsx"
_dst = repo_raw / "online_retail_II.xlsx"
if not _dst.exists() and _src.exists():
    _dst.symlink_to(_src)
    print("  data/raw/: online_retail_II.xlsx symlinked from input.")

# TaFeng — symlink to avoid duplicating a large csv
_src = DATA_ROOT / "tafeng-dataset" / "ta_feng_all_months_merged.csv"
_dst = repo_raw / "ta_feng_all_months_merged.csv"
if not _dst.exists() and _src.exists():
    _dst.symlink_to(_src)
    print("  data/raw/: ta_feng_all_months_merged.csv symlinked from input.")

# Dunnhumby — symlink individual files into data/raw/Dunnhumby datasets/
_dunnh_raw = repo_raw / "Dunnhumby datasets"
_dunnh_raw.mkdir(exist_ok=True)
_dunnh_src = DATA_ROOT / "dunnhumby"
for _fname in ["transaction_data.csv", "hh_demographic.csv", "campaign_table.csv",
               "campaign_desc.csv", "coupon_redempt.csv"]:
    _d = _dunnh_raw / _fname
    _s = _dunnh_src / _fname
    if not _d.exists() and _s.exists():
        _d.symlink_to(_s)
print(f"  data/raw/Dunnhumby datasets/: {sorted(f.name for f in _dunnh_raw.iterdir())}")

# ── 4. Verify GPU (fail fast if incompatible) ─────────────────────────────────
import torch
print(f"\ntorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap  = torch.cuda.get_device_capability(0)
    print(f"GPU  : {name}  (sm_{cap[0]}{cap[1]})")
    print(f"VRAM : {vram:.1f} GB")
    if cap < (7, 0):
        raise RuntimeError(
            f"\n  {name} has CUDA capability sm_{cap[0]}{cap[1]} "
            f"(PyTorch {torch.__version__} requires sm_70+).\n"
            "  Fix: Settings -> Accelerator -> GPU T4 x1 or T4 x2\n"
            "       then restart the kernel and re-run Cell 1."
        )
    print(f"GPU capability OK (sm_{cap[0]}{cap[1]} >= 7.0).")
else:
    raise RuntimeError(
        "No GPU found. Enable one: Settings -> Accelerator -> GPU T4 x1 (or T4 x2)."
    )

# ── 5. Output directory ───────────────────────────────────────────────────────
Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
print("\n/kaggle/working/results/ — ready.")


In [ ]:
import os, subprocess, sys
from pathlib import Path

# Check dataset mounts after the setup cell has populated KAGGLE_DATA_ROOT.
data_root = Path(os.environ.get("KAGGLE_DATA_ROOT", "/kaggle/input"))

REQUIRED_FILES = {
    "cdnow-dataset" : ["CDNOW_master.txt"],
    "uci-retail"    : ["online_retail_II.xlsx"],
    "tafeng-dataset": ["ta_feng_all_months_merged.csv"],
    "dunnhumby"     : ["transaction_data.csv", "hh_demographic.csv"],
}

all_ok = True
for slug, expected in REQUIRED_FILES.items():
    p = data_root / slug
    label = f"{data_root}/{slug}/"
    if not p.exists():
        print(f"MISSING  {label}")
        all_ok = False
        continue
    actual = sorted(f.name for f in p.iterdir())
    missing = [f for f in expected if f not in actual]
    if missing:
        print(f"MISSING  {label}: {missing}")
        print(f"         present: {actual[:10]}")
        all_ok = False
    else:
        print(f"OK       {label}: {expected}")

if not all_ok:
    raise SystemExit("Missing required Kaggle input files. Re-run setup and attach/download the datasets.")

print("\nRunning fixed-protocol preflight (manifest + CDNOW master) ...")
subprocess.run([sys.executable, "validate_final_setup.py", "--smoke"], check=True)

print("\nRunning CDNOW pipeline validation ...")
subprocess.run([sys.executable, "validate_pipelines.py", "--dataset", "cdnow"], check=True)


In [ ]:
from pathlib import Path

RUN_ALL_STAGES = True
FINAL_CONFIG_DIR = "experiments/configs_final"
SEEDS = ["42", "7", "2024"]
MODES = ["sample"]
SKIP_EXISTING = True
HEARTBEAT_INTERVAL = "60"

main_configs = [
    "lstm_base_cdnow_final",
    "lstm_base_uci_final",
    "lstm_base_tafeng_final",
    "lstm_base_dunnhumby_final",
    "lstm_joint_cdnow_final",
    "lstm_joint_uci_final",
    "lstm_joint_tafeng_final",
    "lstm_joint_dunnhumby_final",
    "transformer_joint_cdnow_final",
    "transformer_joint_uci_final",
    "transformer_joint_tafeng_final",
    "transformer_joint_dunnhumby_final",
]

ext3_configs = [
    "extension3_lstm_none_dunnhumby_final",
    "extension3_lstm_static_dunnhumby_final",
    "extension3_lstm_dynamic_dunnhumby_final",
    "extension3_lstm_full_dunnhumby_final",
    "extension3_transformer_none_dunnhumby_final",
    "extension3_transformer_static_dunnhumby_final",
    "extension3_transformer_dynamic_dunnhumby_final",
    "extension3_transformer_full_dunnhumby_final",
]

print(f"Final config dir: {FINAL_CONFIG_DIR}")
print(f"Seeds: {', '.join(SEEDS)}")
print(f"Main sweep: {len(main_configs)} configs x {len(SEEDS)} seeds")
print(f"Extension 3: {len(ext3_configs)} configs x {len(SEEDS)} seeds")


In [ ]:
# Probabilistic benchmark rerun under the same fixed base configs.
# Pareto/NBD is fast and carries final-manifest metadata for fair comparison.
import subprocess, sys

benchmark_configs = {
    "cdnow": "lstm_base_cdnow_final",
    "uci": "lstm_base_uci_final",
    "tafeng": "lstm_base_tafeng_final",
    "dunnhumby": "lstm_base_dunnhumby_final",
}

if RUN_ALL_STAGES:
    for dataset, cfg_name in benchmark_configs.items():
        cfg_path = f"{FINAL_CONFIG_DIR}/{cfg_name}.yaml"
        print(f"\nPareto/NBD benchmark: {dataset} ({cfg_path})")
        subprocess.run(
            [sys.executable, "run_benchmarks.py",
             "--config", cfg_path,
             "--models", "pareto_nbd"],
            check=True,
        )
    print("\nPareto/NBD benchmarks complete.")
else:
    print("Benchmark stage skipped.")


In [ ]:
# Stages 1-3: Base LSTM, Joint LSTM, and Joint Transformer across all datasets.
import subprocess, sys

if RUN_ALL_STAGES:
    cmd = [
        sys.executable, "run_seeds.py",
        "--config_dir", FINAL_CONFIG_DIR,
        "--configs", *main_configs,
        "--seeds", *SEEDS,
        "--modes", *MODES,
        "--heartbeat_interval", HEARTBEAT_INTERVAL,
    ]
    if SKIP_EXISTING:
        cmd.append("--skip_existing")
    subprocess.run(cmd, check=True)
    print("\nStages 1-3 fixed-protocol multi-seed sweep complete.")
else:
    print("Stages 1-3 skipped.")


In [ ]:
# Stage 4: Extension 3 covariate ablation, also 3 seeds for apples-to-apples comparison.
import subprocess, sys

if RUN_ALL_STAGES:
    cmd = [
        sys.executable, "run_seeds.py",
        "--config_dir", FINAL_CONFIG_DIR,
        "--configs", *ext3_configs,
        "--seeds", *SEEDS,
        "--modes", *MODES,
        "--heartbeat_interval", HEARTBEAT_INTERVAL,
    ]
    if SKIP_EXISTING:
        cmd.append("--skip_existing")
    subprocess.run(cmd, check=True)
    print("\nExtension 3 covariate ablation complete.")
else:
    print("Extension 3 skipped.")


In [ ]:
# SHAP covariate attribution for the full Extension 3 models.
# Seed 42 is the canonical attribution checkpoint; performance tables still use all 3 seeds.
import glob, subprocess, sys
from pathlib import Path

Path("/kaggle/working/results/plots").mkdir(parents=True, exist_ok=True)

for model_prefix in [
    "extension3_lstm_full_dunnhumby_final",
    "extension3_transformer_full_dunnhumby_final",
]:
    config_yaml = f"{FINAL_CONFIG_DIR}/{model_prefix}.yaml"
    ckpt_pattern = f"/kaggle/working/results/checkpoints/{model_prefix}*seed42*.pt"
    ckpts = sorted(glob.glob(ckpt_pattern))
    if not ckpts:
        print(f"No checkpoint found: {ckpt_pattern} — run Extension 3 first.")
        continue
    print(f"\nRunning SHAP for {model_prefix} ...")
    subprocess.run([
        sys.executable, "-m", "src.evaluation.shap_analysis",
        "--config", config_yaml,
        "--checkpoint", ckpts[-1],
        "--n_background", "100",
        "--n_explain", "200",
        "--out_dir", "/kaggle/working/results/plots",
    ], check=False)

plots = sorted(Path("/kaggle/working/results/plots").glob("*.png"))
print(f"\nSHAP plots saved: {len(plots)}")
for p in plots:
    print(f"  {p.name}")


In [ ]:
# Build final comparison tables, plots, confidence intervals, and paired bootstrap tests.
import subprocess, sys
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "src.evaluation.compare",
     "--results_dir", "/kaggle/working/results",
     "--latex", "--plots", "--seeds", "--cis",
     "--protocol_variant", "all"],
    check=True,
)

sig_pairs = [
    "lstm_joint_cdnow_final:lstm_base_cdnow_final",
    "transformer_joint_cdnow_final:lstm_joint_cdnow_final",
    "lstm_joint_uci_final:lstm_base_uci_final",
    "transformer_joint_uci_final:lstm_joint_uci_final",
    "lstm_joint_tafeng_final:lstm_base_tafeng_final",
    "transformer_joint_tafeng_final:lstm_joint_tafeng_final",
    "lstm_joint_dunnhumby_final:lstm_base_dunnhumby_final",
    "transformer_joint_dunnhumby_final:lstm_joint_dunnhumby_final",
]
subprocess.run(
    [sys.executable, "-m", "src.evaluation.significance",
     "--metrics_dir", "/kaggle/working/results/tables",
     "--protocol_variant", "all",
     "--seed_filter", "42",
     "--mode_filter", "sample",
     "--pairs", *sig_pairs],
    check=False,
)

plots = sorted(Path("/kaggle/working/results/plots").glob("*.png"))
plots += sorted(Path("/kaggle/working/results/plots").glob("*.pdf"))
print(f"\nPlots generated: {len(plots)}")
for p in plots:
    print(f"  {p.name}")


In [ ]:
# Archive results for download (unchanged from the legacy archive cell).
import shutil
from pathlib import Path

results_dir  = Path("/kaggle/working/results")
archive_stem = "/kaggle/working/results_archive"   # .zip appended automatically

if not results_dir.exists() or not any(results_dir.rglob("*")):
    print("No results found yet — run at least one training cell first.")
else:
    metrics_files = sorted(results_dir.rglob("*_metrics.json"))
    ckpt_files    = sorted(results_dir.rglob("*.pt"))
    print(f"Metrics files  : {len(metrics_files)}")
    print(f"Checkpoints    : {len(ckpt_files)}")
    for f in metrics_files:
        print(f"  {f.name}")
    shutil.make_archive(archive_stem, "zip", results_dir)
    archive_path = Path(archive_stem + ".zip")
    size_mb = archive_path.stat().st_size / (1024 ** 2)
    print(f"\nArchive created : {archive_path}  ({size_mb:.1f} MB)")
    print("Download via    : Kaggle notebook -> Output tab -> results_archive.zip")
